# LER-PUU MKRI — Preprocessing: Doccano JSONL → skema BIO → Sentence-Split CoNLL

Pipeline ini mengkonversi hasil anotasi Doccano (format JSONL) ke format BIO dan menjadi file CoNLL, yang siap digunakan untuk training CRF dan BiLSTM-CRF.

**Fitur utama:**
- Split dokumen panjang menjadi kalimat pendek (max 128 token)
- Aman untuk label long-span: DICTUM, LAW_NAME, LAW_CONS_ART tidak terpotong di tengah
- Audit konsistensi anotasi sebelum konversi
- Output: `train.conll`, `dev.conll`, `test.conll` siap training

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Cell 1 — Import & Konfigurasi

In [ ]:
import json
import os
import re
import random
from collections import Counter, defaultdict
from pathlib import Path

# ── Konfigurasi path ───────────────────────────────────────────────────────────
# Sesuaikan path ke file JSONL hasil export Doccano
JSONL_PATH   = '/content/drive/MyDrive/Output_doccano100_v5.jsonl'   # file export Doccano (semua dokumen)
OUTPUT_DIR   = '/content/drive/MyDrive/data_conll_100_v5'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Output akan disimpan ke: {OUTPUT_DIR}')
                   # folder output CoNLL

# ── Parameter split kalimat ────────────────────────────────────────────────────
MAX_SEQ_LEN  = 128     # panjang maksimum kalimat (token)
                       # BiLSTM-CRF optimal: 64–256
                       # terlalu pendek (< 64): DICTUM terpotong banyak
                       # terlalu panjang (> 256): loss masalah kembali

# ── Parameter split dataset ────────────────────────────────────────────────────
TRAIN_RATIO  = 0.70
DEV_RATIO    = 0.15
TEST_RATIO   = 0.15
RANDOM_SEED  = 42

# ── Label long-span yang TIDAK boleh dipotong di tengah ───────────────────────
# Kalimat hanya boleh dipotong jika label aktif BUKAN salah satu dari ini
# Jika sedang di tengah DICTUM, terus sampai kalimat (titik/semicolon) selesai
PROTECTED_LABELS = {
    'DICTUM', 'LAW_NAME', 'LAW_CONS_ART', 'LEGAL_DOC', 'LEGAL_DOC_NUM'
}

os.makedirs(OUTPUT_DIR, exist_ok=True)
random.seed(RANDOM_SEED)

print('✅ Konfigurasi selesai')
print(f'   Input  : {JSONL_PATH}')
print(f'   Output : {OUTPUT_DIR}/')
print(f'   Max seq len : {MAX_SEQ_LEN} token')
print(f'   Split  : {TRAIN_RATIO:.0%} / {DEV_RATIO:.0%} / {TEST_RATIO:.0%}')

## Cell 2 — Load & Parse JSONL Doccano

In [ ]:
def load_doccano_jsonl(filepath):
    """
    Load JSONL Doccano — support semua format:
      - Format lama : {"label":  [[start, end, "LABEL"], ...]}
      - Format lama : {"labels": [[start, end, "LABEL"], ...]}
      - Format baru : {"entities": [{"start_offset":s, "end_offset":e, "label":"LABEL"}, ...]}
      - Format baru : {"entities": [{"start_offset":s, "end_offset":e, "type":"LABEL"}, ...]}
    """
    docs = []
    with open(filepath, encoding='utf-8') as f:
        for line_no, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError as e:
                print(f'  ⚠ Baris {line_no}: JSON error — {e}')
                continue

            text   = obj.get('text', '')
            doc_id = obj.get('id', line_no)

            if not text:
                print(f'  ⚠ Dokumen {doc_id}: teks kosong, dilewati')
                continue

            # ── Deteksi format span secara otomatis ───────────────────────────
            normalized = []

            # Format baru Doccano v2: key 'entities' berisi list of dict
            if 'entities' in obj and obj['entities']:
                for ent in obj['entities']:
                    if isinstance(ent, dict):
                        # Key offset bisa 'start_offset'/'end_offset'
                        # atau 'start'/'end' tergantung versi
                        s = ent.get('start_offset', ent.get('start'))
                        e = ent.get('end_offset',   ent.get('end'))
                        # Key label bisa 'label' atau 'type'
                        lbl = ent.get('label', ent.get('type', ''))
                        if s is not None and e is not None and lbl:
                            normalized.append((int(s), int(e), str(lbl)))
                    elif isinstance(ent, list) and len(ent) == 3:
                        # Kadang entities berisi list juga
                        normalized.append((int(ent[0]), int(ent[1]), str(ent[2])))

            # Format lama: key 'label' atau 'labels' berisi list of list
            elif 'label' in obj or 'labels' in obj:
                raw = obj.get('label', obj.get('labels', []))
                for span in raw:
                    if isinstance(span, list) and len(span) == 3:
                        normalized.append((int(span[0]), int(span[1]), str(span[2])))

            # Urutkan berdasarkan posisi awal
            normalized.sort(key=lambda x: x[0])

            docs.append({'id': doc_id, 'text': text, 'labels': normalized})

    return docs


# ── Reload dengan fungsi yang sudah diperbaiki ────────────────────────────────
docs = load_doccano_jsonl(JSONL_PATH)

# Verifikasi: cari dokumen pertama yang punya anotasi
n_with_labels = sum(1 for d in docs if d['labels'])
print(f'Total dokumen     : {len(docs)}')
print(f'Dokumen beranotasi: {n_with_labels}')

for d in docs[:10]:
    if d['labels']:
        print(f'\nDokumen id={d["id"]} — {len(d["labels"])} span:')
        for span in d['labels'][:5]:
            s, e, lbl = span
            print(f'  [{s}:{e}] {lbl:<20} → "{d["text"][s:e][:60]}"')
        break

## Cell 3 — Tokenisasi & Konversi ke BIO per Dokumen

In [ ]:
# ── DIAGNOSTIC: cek format span vs posisi karakter teks ───────────────────────
doc = docs[0]

print(f"Panjang teks     : {len(doc['text'])} karakter")
print(f"Jumlah span      : {len(doc['labels'])}")
print(f"\n5 span pertama:")
for span in doc['labels'][:5]:
    s, e, lbl = span
    snippet = doc['text'][s:e]
    print(f"  [{s}:{e}] {lbl:<20} → '{snippet[:60]}'")

print(f"\n5 span terakhir:")
for span in doc['labels'][-5:]:
    s, e, lbl = span
    snippet = doc['text'][s:e]
    print(f"  [{s}:{e}] {lbl:<20} → '{snippet[:60]}'")

# Cek token pertama yang seharusnya match dengan span
print(f"\n--- Token vs Span overlap check ---")
toks = tokenize_with_offsets(doc['text'])
print(f"Token ke-0: '{toks[0][0]}' offset [{toks[0][1]}:{toks[0][2]}]")
print(f"Token ke-1: '{toks[1][0]}' offset [{toks[1][1]}:{toks[1][2]}]")
print(f"Token ke-5: '{toks[5][0]}' offset [{toks[5][1]}:{toks[5][2]}]")

# Cari token pertama yang seharusnya kena span
if doc['labels']:
    first_span = doc['labels'][0]
    s, e, lbl = first_span
    print(f"\nSpan pertama [{s}:{e}] '{lbl}'")
    print(f"Teks di span: '{doc['text'][s:e][:80]}'")

    # Cari token mana yang overlap
    matching = [(i,t,ts,te) for i,(t,ts,te) in enumerate(toks)
                if ts < e and te > s]
    print(f"Token yang overlap dengan span ini: {len(matching)}")
    for i,t,ts,te in matching[:5]:
        print(f"  Token {i}: '{t}' [{ts}:{te}]")

In [ ]:
def tokenize_with_offsets(text):
    """
    Tokenisasi teks menjadi (token, char_start, char_end).
    Menggunakan split berbasis whitespace + tanda baca sebagai token terpisah.
    Mempertahankan offset karakter asli untuk mapping ke span anotasi.
    """
    tokens_with_offsets = []
    # Regex: pisahkan token berdasarkan whitespace, tanda baca tetap sebagai token
    # Pola ini memisahkan tanda baca dari kata tapi tidak membuangnya
    pattern = re.compile(r'\S+')

    for match in pattern.finditer(text):
        token = match.group()
        start = match.start()
        end   = match.end()
        tokens_with_offsets.append((token, start, end))

    return tokens_with_offsets


def assign_bio_labels(tokens_with_offsets, spans):
    """
    Assign label BIO ke setiap token berdasarkan span anotasi.

    Aturan:
    - Token yang char_start-nya berada tepat di awal span → B-LABEL
    - Token yang berada di dalam span (bukan yang pertama) → I-LABEL
    - Token di luar semua span → O

    Untuk span yang overlap (seharusnya tidak ada di Doccano),
    span dengan prioritas lebih tinggi (index lebih kecil) yang dipakai.

    Args:
        tokens_with_offsets: list of (token_str, char_start, char_end)
        spans: list of (span_start, span_end, label_str) — sudah diurutkan

    Return:
        list of (token_str, bio_label)
    """
    result = []
    span_idx = 0
    n_spans  = len(spans)

    for token, tok_start, tok_end in tokens_with_offsets:
        label = 'O'

        # Maju ke span yang relevan
        while span_idx < n_spans and spans[span_idx][1] <= tok_start:
            span_idx += 1

        # Cek apakah token ini di dalam span aktif
        for si in range(span_idx, n_spans):
            s_start, s_end, s_label = spans[si]
            if s_start >= tok_end:
                break  # span di sebelah kanan token ini

            # Overlap: token_start < s_end AND s_start < token_end
            if s_start < tok_end and s_end > tok_start:
                if tok_start <= s_start:
                    # Token dimulai sebelum atau tepat di awal span → B-
                    # (ini adalah token pertama dalam span)
                    label = f'B-{s_label}'
                else:
                    # Token di dalam span → I-
                    label = f'I-{s_label}'
                break  # pakai span pertama yang match

        result.append((token, label))

    return result


def doc_to_bio(doc):
    """
    Konversi satu dokumen Doccano → list of (token, bio_label).
    Return: list of (str, str)
    """
    toks = tokenize_with_offsets(doc['text'])
    bio  = assign_bio_labels(toks, doc['labels'])
    return bio


# ── Test pada dokumen pertama ──────────────────────────────────────────────────
if docs:
    sample_bio = doc_to_bio(docs[0])
    print(f'Dokumen 0: {len(sample_bio)} token')

    # Tampilkan 20 token pertama
    print('\n50 token pertama:')
    for tok, lbl in sample_bio[:50]:
        print(f'  {tok:<30} {lbl}')

    # Tampilkan distribusi label
    label_dist = Counter(lbl for _, lbl in sample_bio)
    print(f'\nDistribusi label dokumen 0:')
    for lbl, cnt in sorted(label_dist.items()):
        print(f'  {lbl:<25}: {cnt}')

## Cell 4 — Sentence Splitting yang Aman untuk Long-Span Label

In [ ]:
# Penanda batas kalimat dalam teks hukum Indonesia
# Titik koma (;) sangat umum di amar putusan MK
SENTENCE_END_TOKENS = {'.', ';', '?', '!'}

# Token yang jika muncul setelah batas kalimat, memulai kalimat baru
# (nomor butir amar: 1. 2. dst, huruf kapital awal kalimat)
AMAR_NUMBER_RE = re.compile(r'^\d+\.?$|^[IVX]+\.$')


def get_base_label(bio_label):
    """Ekstrak nama label tanpa B-/I- prefix. 'B-DICTUM' → 'DICTUM', 'O' → 'O'"""
    if bio_label.startswith('B-') or bio_label.startswith('I-'):
        return bio_label[2:]
    return bio_label


def is_protected(bio_label):
    """Cek apakah label termasuk label yang dilindungi (tidak boleh dipotong)."""
    return get_base_label(bio_label) in PROTECTED_LABELS


def split_doc_into_sentences(bio_sequence, max_len=MAX_SEQ_LEN):
    """
    Memecah satu dokumen (list of (token, label)) menjadi kalimat-kalimat pendek.

    Strategi prioritas (dari tinggi ke rendah):

    1. BATAS LUNAK: potong di tanda baca kalimat (. ; ! ?)
       HANYA jika token BERIKUTNYA bukan I-LABEL dari protected span.
       Ini memastikan DICTUM, LAW_NAME, dll. tidak dipotong di tengah klausa.

    2. BATAS KERAS: jika kalimat sudah melebihi max_len*2 token dan
       saat ini TIDAK berada di tengah protected span, potong paksa.

    3. BATAS MUTLAK: jika kalimat melebihi max_len*3 (dokumen patologis),
       potong paksa meski di tengah protected span — dan log peringatan.

    Return: list of list of (token, label)
    """
    sentences  = []
    current    = []
    n          = len(bio_sequence)
    hard_limit = max_len * 3   # potong paksa jika > 3× max

    i = 0
    while i < n:
        token, label = bio_sequence[i]
        current.append((token, label))

        # ── Cek kondisi pemotongan ─────────────────────────────────────────────
        at_sentence_end = token in SENTENCE_END_TOKENS
        cur_len         = len(current)

        # Tentukan apakah token BERIKUTNYA memulai I- dari protected label
        next_is_continuation = False
        if i + 1 < n:
            _, next_label = bio_sequence[i + 1]
            next_is_continuation = next_label.startswith('I-') and is_protected(next_label)

        # Tentukan apakah token SAAT INI berada di dalam protected span
        currently_in_protected = is_protected(label)

        should_cut = False
        cut_reason = ''

        if cur_len >= hard_limit:
            # Batas mutlak — potong paksa apapun kondisinya
            should_cut = True
            cut_reason = f'hard_limit({hard_limit})'

        elif at_sentence_end and not next_is_continuation and cur_len >= 5:
            # Batas lunak — potong di tanda baca, aman untuk semua label
            should_cut = True
            cut_reason = 'sentence_end'

        elif cur_len >= max_len * 2 and not currently_in_protected:
            # Batas keras — terlalu panjang tapi tidak di protected span
            should_cut = True
            cut_reason = f'soft_limit({max_len*2})'

        elif cur_len >= max_len and at_sentence_end and not currently_in_protected:
            # Panjang sudah cukup dan ada batas kalimat di non-protected
            should_cut = True
            cut_reason = 'max_len+sentence_end'

        if should_cut and current:
            sentences.append(current)
            current = []

        i += 1

    # Sisa token
    if current:
        sentences.append(current)

    return sentences


def fix_bio_continuity(sentence):
    """
    Perbaiki inkonsistensi BIO setelah pemotongan:
    Jika sebuah kalimat dimulai dengan I-LABEL (bukan B-LABEL),
    ubah token pertama menjadi B-LABEL.

    Ini terjadi ketika pemotongan terpaksa memotong di tengah span.
    """
    if not sentence:
        return sentence

    result = []
    prev_label = 'O'

    for idx, (token, label) in enumerate(sentence):
        if label.startswith('I-'):
            base = label[2:]
            # Jika I- muncul setelah O atau B-/I- label berbeda → ubah ke B-
            if prev_label == 'O' or get_base_label(prev_label) != base:
                label = f'B-{base}'
        result.append((token, label))
        prev_label = label

    return result


# ── Test pada dokumen pertama ──────────────────────────────────────────────────
if docs:
    sample_bio   = doc_to_bio(docs[0])
    sample_sents = split_doc_into_sentences(sample_bio, max_len=MAX_SEQ_LEN)
    sample_sents = [fix_bio_continuity(s) for s in sample_sents]

    lengths = [len(s) for s in sample_sents]
    print(f'Dokumen 0:')
    print(f'  Sebelum split : {len(sample_bio)} token (1 unit)')
    print(f'  Setelah split : {len(sample_sents)} kalimat')
    print(f'  Panjang rata2 : {sum(lengths)/len(lengths):.1f} token')
    print(f'  Panjang max   : {max(lengths)} token')
    print(f'  Panjang min   : {min(lengths)} token')
    print()

    # Tampilkan kalimat yang mengandung DICTUM
    dictum_sents = [(i, s) for i, s in enumerate(sample_sents)
                    if any('DICTUM' in lbl for _, lbl in s)]
    print(f'  Kalimat dengan DICTUM: {len(dictum_sents)}')
    for i, sent in dictum_sents[:2]:
        print(f'  Kalimat {i} ({len(sent)} token):')
        for tok, lbl in sent[:8]:
            print(f'    {tok:<25} {lbl}')
        if len(sent) > 8:
            print(f'    ... ({len(sent)-8} token lagi)')

## Cell 5 — Konversi Semua Dokumen & Audit Kualitas

In [ ]:
def process_all_docs(docs, max_len=MAX_SEQ_LEN):
    """
    Konversi semua dokumen Doccano ke list of sentences.
    Setiap sentence = list of (token, bio_label).

    Return:
        all_sentences : list of list of (token, label)
        doc_map       : list of (doc_id, sent_start_idx, sent_end_idx)
                        untuk tracing kalimat ke dokumen asal
    """
    all_sentences = []
    doc_map       = []
    stats = {
        'total_docs'  : len(docs),
        'total_tokens_before': 0,
        'total_sents' : 0,
        'long_sents'  : 0,   # kalimat > max_len
        'forced_cuts' : 0,   # pemotongan paksa di protected span
    }

    for doc in docs:
        bio = doc_to_bio(doc)
        stats['total_tokens_before'] += len(bio)

        sents = split_doc_into_sentences(bio, max_len=max_len)
        sents = [fix_bio_continuity(s) for s in sents]

        sent_start = len(all_sentences)
        for sent in sents:
            all_sentences.append(sent)
            if len(sent) > max_len:
                stats['long_sents'] += 1
        sent_end = len(all_sentences)

        doc_map.append((doc['id'], sent_start, sent_end))
        stats['total_sents'] += len(sents)

    return all_sentences, doc_map, stats


all_sents, doc_map, proc_stats = process_all_docs(docs, max_len=MAX_SEQ_LEN)

# ── Statistik hasil ───────────────────────────────────────────────────────────
sent_lengths = [len(s) for s in all_sents]
print('='*55)
print('  HASIL PREPROCESSING')
print('='*55)
print(f'  Total dokumen          : {proc_stats["total_docs"]}')
print(f'  Total token (sebelum)  : {proc_stats["total_tokens_before"]:,}')
print(f'  Total kalimat (sesudah): {len(all_sents):,}')
print(f'  Rata-rata panjang      : {sum(sent_lengths)/len(sent_lengths):.1f} token')
print(f'  Median panjang         : {sorted(sent_lengths)[len(sent_lengths)//2]} token')
print(f'  Max panjang            : {max(sent_lengths)} token')
print(f'  Min panjang            : {min(sent_lengths)} token')
print(f'  Kalimat > {MAX_SEQ_LEN} token     : {proc_stats["long_sents"]}')

print(f'\nDistribusi panjang kalimat:')
bins = [0, 32, 64, 128, 256, 512, 9999]
for i in range(len(bins)-1):
    count = sum(1 for l in sent_lengths if bins[i] <= l < bins[i+1])
    pct   = count / len(sent_lengths) * 100
    bar   = '█' * int(pct / 2)
    label = f'{bins[i]:>4}–{bins[i+1] if bins[i+1]<9999 else "∞":>4}'
    print(f'  {label}: {count:5d} ({pct:5.1f}%) {bar}')

# ── Distribusi label ──────────────────────────────────────────────────────────
print(f'\nDistribusi label (B- saja):')
all_labels_flat = [lbl for sent in all_sents for _, lbl in sent]
label_cnt = Counter(all_labels_flat)
b_labels  = {k: v for k, v in label_cnt.items() if k.startswith('B-')}
for lbl, cnt in sorted(b_labels.items(), key=lambda x: -x[1]):
    print(f'  {lbl:<25}: {cnt:5d}')
print(f'  {"O":<25}: {label_cnt["O"]:5d}')

## Cell 6 — Audit Konsistensi BIO (DICTUM & Label Lain)

In [ ]:
def audit_bio_consistency(sentences, label_name='DICTUM'):
    """
    Audit konsistensi label tertentu di seluruh dataset:
    1. Kalimat yang I- tanpa B- sebelumnya (seharusnya sudah di-fix)
    2. Span yang terpotong (B- di akhir kalimat tanpa I- di kalimat berikutnya)
    3. Kalimat kosong atau token tunggal
    """
    b_tag = f'B-{label_name}'
    i_tag = f'I-{label_name}'

    issues = []
    label_sents = 0

    for si, sent in enumerate(sentences):
        labels = [lbl for _, lbl in sent]
        has_label = any(l in (b_tag, i_tag) for l in labels)
        if has_label:
            label_sents += 1

        # Cek I- tanpa B- sebelumnya dalam kalimat yang sama
        prev = 'O'
        for ti, lbl in enumerate(labels):
            if lbl == i_tag and get_base_label(prev) != label_name:
                issues.append(f'Sent {si:4d} pos {ti:3d}: {i_tag} tanpa {b_tag} sebelumnya')
            prev = lbl

        # Kalimat terlalu pendek
        if len(sent) == 0:
            issues.append(f'Sent {si}: kalimat kosong')

    print(f'\n── Audit {label_name} ────────────────────────────')
    print(f'   Kalimat dengan {label_name:<15}: {label_sents}')
    print(f'   Issues ditemukan              : {len(issues)}')
    for iss in issues[:5]:
        print(f'   ⚠ {iss}')
    if not issues:
        print(f'   ✅ Semua label {label_name} konsisten')
    return issues


# Audit label-label penting
print('='*55)
print('  AUDIT KONSISTENSI BIO')
print('='*55)

for lbl in ['DICTUM', 'LAW_NAME', 'LAW_CONS_ART', 'PERSON', 'JUDGE']:
    audit_bio_consistency(all_sents, label_name=lbl)

# Cek kalimat terlalu panjang yang masih ada
very_long = [(i, len(s)) for i, s in enumerate(all_sents) if len(s) > MAX_SEQ_LEN]
print(f'\n── Kalimat > {MAX_SEQ_LEN} token: {len(very_long)}')
for si, length in very_long[:5]:
    labels_in = set(get_base_label(l) for _, l in all_sents[si] if l != 'O')
    print(f'   Sent {si}: {length} token | label: {labels_in}')

## Cell 7 — Split Train / Dev / Test Berbasis Dokumen

In [ ]:
def split_by_document(docs, doc_map, all_sents,
                       train_ratio=TRAIN_RATIO,
                       dev_ratio=DEV_RATIO,
                       seed=RANDOM_SEED):
    """
    Split dataset di level DOKUMEN (bukan kalimat).
    Ini memastikan kalimat dari dokumen yang sama tidak tersebar
    di train dan test sekaligus — mencegah data leakage.

    Return:
        train_sents, dev_sents, test_sents : masing-masing list of sentences
    """
    n_docs = len(docs)
    indices = list(range(n_docs))
    random.seed(seed)
    random.shuffle(indices)

    n_train = int(n_docs * train_ratio)
    n_dev   = int(n_docs * dev_ratio)

    train_doc_ids = set(indices[:n_train])
    dev_doc_ids   = set(indices[n_train:n_train + n_dev])
    test_doc_ids  = set(indices[n_train + n_dev:])

    train_sents, dev_sents, test_sents = [], [], []

    for doc_idx, (doc_id, sent_start, sent_end) in enumerate(doc_map):
        doc_sentences = all_sents[sent_start:sent_end]
        if doc_idx in train_doc_ids:
            train_sents.extend(doc_sentences)
        elif doc_idx in dev_doc_ids:
            dev_sents.extend(doc_sentences)
        else:
            test_sents.extend(doc_sentences)

    return train_sents, dev_sents, test_sents


train_sents, dev_sents, test_sents = split_by_document(
    docs, doc_map, all_sents
)

# ── Verifikasi split ──────────────────────────────────────────────────────────
print('='*55)
print('  HASIL SPLIT DATASET')
print('='*55)

for name, sents in [('Train', train_sents), ('Dev', dev_sents), ('Test', test_sents)]:
    lengths  = [len(s) for s in sents]
    n_tokens = sum(lengths)
    b_counts = Counter(
        lbl for sent in sents for _, lbl in sent if lbl.startswith('B-')
    )
    print(f'\n  {name}:')
    print(f'    Kalimat : {len(sents):5d}')
    print(f'    Token   : {n_tokens:7,}')
    print(f'    Avg len : {sum(lengths)/len(lengths):.1f}')
    print(f'    DICTUM  : {b_counts.get("B-DICTUM", 0):5d} entitas')
    print(f'    PERSON  : {b_counts.get("B-PERSON", 0):5d} entitas')
    print(f'    JUDGE   : {b_counts.get("B-JUDGE", 0):5d} entitas')
    print(f'    LAW_NAME: {b_counts.get("B-LAW_NAME", 0):5d} entitas')

## Cell 8 — Tulis File CoNLL

In [ ]:
def write_conll(sentences, filepath):
    """
    Tulis sentences ke file CoNLL.
    Format: satu token per baris (token<spasi>label),
    kalimat dipisahkan baris kosong.
    """
    with open(filepath, 'w', encoding='utf-8') as f:
        for sent in sentences:
            if not sent:
                continue
            for token, label in sent:
                # Sanitasi token: hapus karakter newline jika ada
                token_clean = token.replace('\n', ' ').replace('\r', ' ').strip()
                if not token_clean:
                    token_clean = '_'
                f.write(f'{token_clean} {label}\n')
            f.write('\n')   # baris kosong = batas kalimat

    n_tokens = sum(len(s) for s in sentences)
    print(f'  ✅ {filepath}')
    print(f'     {len(sentences)} kalimat | {n_tokens:,} token')


print('Menulis file CoNLL...')
write_conll(train_sents, os.path.join(OUTPUT_DIR, 'train.conll'))
write_conll(dev_sents,   os.path.join(OUTPUT_DIR, 'dev.conll'))
write_conll(test_sents,  os.path.join(OUTPUT_DIR, 'test.conll'))

print(f'\n✅ Semua file tersimpan di folder: {OUTPUT_DIR}/')

## Cell 9 — Verifikasi Final: Baca Ulang & Bandingkan

In [ ]:
def read_conll(filepath):
    """Baca file CoNLL → list of (tokens, labels)."""
    sentences = []
    tokens, labels = [], []
    with open(filepath, encoding='utf-8') as f:
        for line in f:
            line = line.rstrip()
            if line == '' or line.startswith('-DOCSTART-'):
                if tokens:
                    sentences.append((tokens, labels))
                    tokens, labels = [], []
            else:
                parts = line.split()
                tokens.append(parts[0])
                labels.append(parts[-1])
    if tokens:
        sentences.append((tokens, labels))
    return sentences


# Baca ulang dan verifikasi
print('Verifikasi file CoNLL yang dihasilkan...')
print('='*55)

for split, fname in [('Train','train.conll'), ('Dev','dev.conll'), ('Test','test.conll')]:
    path = os.path.join(OUTPUT_DIR, fname)
    data = read_conll(path)

    lengths   = [len(t) for t, _ in data]
    all_lbls  = [l for _, labels in data for l in labels]
    b_counts  = Counter(l for l in all_lbls if l.startswith('B-'))

    print(f'\n  [{split}] {fname}')
    print(f'    Kalimat      : {len(data):5d}')
    print(f'    Token        : {len(all_lbls):7,}')
    print(f'    Avg seq len  : {sum(lengths)/len(lengths):6.1f}')
    print(f'    Max seq len  : {max(lengths):6d}')
    print(f'    Token O      : {Counter(all_lbls)["O"]:7,} ({Counter(all_lbls)["O"]/len(all_lbls)*100:.1f}%)')

    # Top-5 entity label
    print(f'    Top entity:')
    for lbl, cnt in b_counts.most_common(5):
        print(f'      {lbl:<22}: {cnt}')

# ── Cek DICTUM tidak hilang ────────────────────────────────────────────────────
print('\n── Sanity check DICTUM ───────────────────────────')
for split, fname in [('Train','train.conll'),('Dev','dev.conll'),('Test','test.conll')]:
    data = read_conll(os.path.join(OUTPUT_DIR, fname))
    n_dictum = sum(1 for _, lbls in data for l in lbls if l == 'B-DICTUM')
    n_sents_with = sum(1 for _, lbls in data if 'B-DICTUM' in lbls)
    print(f'  {split:<6}: {n_dictum:3d} entitas DICTUM | {n_sents_with:3d} kalimat mengandung DICTUM')

## Cell 10 — Ringkasan & Langkah Selanjutnya

Setelah cell ini selesai tanpa error, lanjutkan ke notebook training utama dengan mengganti path data:

```python
# Di notebook training (LER_Baseline_MKRI.ipynb), Cell 7:
train_data = read_conll('./data/train.conll')
dev_data   = read_conll('./data/dev.conll')
test_data  = read_conll('./data/test.conll')
```

**Ekspektasi perubahan loss BiLSTM-CRF setelah re-segmentasi:**

| Kondisi | Train Loss Epoch 1 |
|---|---|
| Sebelum (avg 1372 tok) | ~4500–5000 |
| Setelah (avg ~50 tok)  | ~10–50     |

Loss yang normal memungkinkan F1 mulai naik di epoch 5–15.

In [ ]:
print('='*55)
print('  PREPROCESSING SELESAI')
print('='*55)

train_data_check = read_conll(os.path.join(OUTPUT_DIR, 'train.conll'))
avg_len = sum(len(t) for t,_ in train_data_check) / len(train_data_check)

print(f'  Rata-rata panjang kalimat train : {avg_len:.1f} token')

if avg_len > 300:
    print(f'  ⚠ Masih terlalu panjang — coba turunkan MAX_SEQ_LEN ke 64')
elif avg_len > 150:
    print(f'  ⚠ Cukup panjang — training bisa lambat, pertimbangkan MAX_SEQ_LEN=64')
else:
    print(f'  ✅ Panjang kalimat optimal untuk BiLSTM-CRF')

print(f'\nFile siap:')
for fname in ['train.conll', 'dev.conll', 'test.conll']:
    fpath = os.path.join(OUTPUT_DIR, fname)
    size  = os.path.getsize(fpath) / 1024
    print(f'  {fpath}  ({size:.0f} KB)')